# Matrix Operations - Exercises

**Chapter 02, Section 02** | [notes.md](notes.md) | [theory.ipynb](theory.ipynb)

This notebook is the practice counterpart to the `Matrix Operations` chapter.

The exercise path is ordered for learning:
1. matrix shapes and special structure
2. elementwise operations, trace, and transpose
3. matrix multiplication and its different interpretations
4. dense layers and attention as matrix pipelines
5. inverse, conditioning, and solving systems
6. pseudo-inverse and least squares
7. determinant and elimination structure
8. LU, QR, eigendecomposition, and SVD
9. Cholesky, low rank, and AI-specific compression intuition

This notebook is intentionally not a shallow coding worksheet. Many exercises include a short written or proof-style component before the implementation.

Each exercise has:
1. a problem statement
2. a scaffold cell with TODOs
3. a complete solution cell

The goal is not only to get the numeric answer, but to become fluent at reading matrix formulas the way they appear in linear algebra, scientific computing, and deep learning.

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)


def softmax_rows(S):
    S = np.asarray(S, dtype=float)
    S = S - S.max(axis=1, keepdims=True)
    exp_S = np.exp(S)
    return exp_S / exp_S.sum(axis=1, keepdims=True)


def outer_sum_product(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    m, p = A.shape
    p2, n = B.shape
    if p != p2:
        raise ValueError("inner dimensions must match")
    C = np.zeros((m, n), dtype=float)
    for k in range(p):
        C += np.outer(A[:, k], B[k, :])
    return C


def forward_substitution(L, b):
    L = np.asarray(L, dtype=float)
    b = np.asarray(b, dtype=float)
    y = np.zeros_like(b, dtype=float)
    for i in range(len(b)):
        y[i] = (b[i] - np.dot(L[i, :i], y[:i])) / L[i, i]
    return y


def back_substitution(U, y):
    U = np.asarray(U, dtype=float)
    y = np.asarray(y, dtype=float)
    x = np.zeros_like(y, dtype=float)
    for i in range(len(y) - 1, -1, -1):
        x[i] = (y[i] - np.dot(U[i, i + 1:], x[i + 1:])) / U[i, i]
    return x


def lu_no_pivot(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    U = A.copy()
    for k in range(n - 1):
        pivot = U[k, k]
        if abs(pivot) < 1e-12:
            raise ValueError("zero pivot encountered; pivoting required")
        for i in range(k + 1, n):
            factor = U[i, k] / pivot
            L[i, k] = factor
            U[i, k:] -= factor * U[k, k:]
    return L, U


---
## Exercise 1: Shapes, Rows, Columns, and Special Matrices

**Task**: Build fluency with the basic objects before doing any heavy algebra.

**Requirements**:
- Create one rectangular matrix, one identity matrix, one diagonal matrix, and one symmetric matrix.
- Write helpers `row(A, i)` and `col(A, j)`.
- Verify which matrices are square and which are symmetric.

**Written part**:
- Explain why only square matrices can even be candidates for ordinary inversion.

In [ ]:
# === Exercise 1: Scaffold ===

def row(A, i):
    pass  # YOUR CODE HERE


def col(A, j):
    pass  # YOUR CODE HERE


def is_square(A):
    pass  # YOUR CODE HERE


def is_symmetric(A):
    pass  # YOUR CODE HERE


# Build example matrices and test the helpers.

In [ ]:
# === Exercise 1: Solution ===

def row(A, i):
    return np.asarray(A)[i, :]


def col(A, j):
    return np.asarray(A)[:, j]


def is_square(A):
    A = np.asarray(A)
    return A.shape[0] == A.shape[1]


def is_symmetric(A):
    A = np.asarray(A, dtype=float)
    return is_square(A) and np.allclose(A, A.T)


rect = np.array([[1., 2., 3.], [4., 5., 6.]])
I = np.eye(3)
D = np.diag([2., -1., 4.])
S = np.array([[2., 1., 3.], [1., 4., 5.], [3., 5., 6.]])

print("row(rect, 1) =", row(rect, 1))
print("col(rect, 2) =", col(rect, 2))
print("is_square(rect) =", is_square(rect))
print("is_square(I) =", is_square(I))
print("is_symmetric(D) =", is_symmetric(D))
print("is_symmetric(S) =", is_symmetric(S))

---
## Exercise 2: Elementwise Operations, Trace, and Transpose

**Task**: Implement the basic non-GEMM matrix operations directly.

**Requirements**:
- Implement matrix addition and Hadamard product without using `+` or `*` on whole arrays.
- Implement `trace_manual(A)`.
- Verify `(AB)^T = B^T A^T` on one concrete example.

**Written part**:
- Explain why transpose reverses order but addition does not.

In [ ]:
# === Exercise 2: Scaffold ===

def add_manual(A, B):
    pass  # YOUR CODE HERE


def hadamard_manual(A, B):
    pass  # YOUR CODE HERE


def trace_manual(A):
    pass  # YOUR CODE HERE


# Test these on at least one 2x2 example and one transpose identity example.

In [ ]:
# === Exercise 2: Solution ===

def add_manual(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    C = np.zeros_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            C[i, j] = A[i, j] + B[i, j]
    return C


def hadamard_manual(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    C = np.zeros_like(A)
    for i in range(A.shape[0]):
        for j in range(A.shape[1]):
            C[i, j] = A[i, j] * B[i, j]
    return C


def trace_manual(A):
    A = np.asarray(A, dtype=float)
    total = 0.0
    for i in range(A.shape[0]):
        total += A[i, i]
    return total


A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])
print("manual A+B =\n", add_manual(A, B))
print("manual A odot B =\n", hadamard_manual(A, B))
print("trace_manual(A) =", trace_manual(A))
print("trace numpy(A) =", np.trace(A))
print("(AB)^T == B^T A^T ?", np.allclose((A @ B).T, B.T @ A.T))

---
## Exercise 3: Matrix Multiplication by Formula and by Outer Products

**Task**: Implement matrix multiplication manually and then reproduce the same result as a sum of outer products.

**Requirements**:
- Implement `matmul_manual(A, B)` with three loops.
- Implement `matmul_outer(A, B)` using outer products of columns/rows.
- Verify both agree with `A @ B`.

**Written part**:
- Explain why `rank(AB)` cannot exceed either `rank(A)` or `rank(B)`.

In [ ]:
# === Exercise 3: Scaffold ===

def matmul_manual(A, B):
    pass  # YOUR CODE HERE


def matmul_outer(A, B):
    pass  # YOUR CODE HERE


# Use a non-square example where inner dimensions match.

In [ ]:
# === Exercise 3: Solution ===

def matmul_manual(A, B):
    A = np.asarray(A, dtype=float)
    B = np.asarray(B, dtype=float)
    m, p = A.shape
    p2, n = B.shape
    if p != p2:
        raise ValueError("inner dimensions must match")
    C = np.zeros((m, n), dtype=float)
    for i in range(m):
        for j in range(n):
            for k in range(p):
                C[i, j] += A[i, k] * B[k, j]
    return C


def matmul_outer(A, B):
    return outer_sum_product(A, B)


A = np.array([[1., 2., 3.], [4., 5., 6.]])
B = np.array([[1., 0.], [0., 1.], [1., 1.]])
C_manual = matmul_manual(A, B)
C_outer = matmul_outer(A, B)
C_np = A @ B
print("manual =\n", C_manual)
print("outer-sum =\n", C_outer)
print("numpy =\n", C_np)
print("manual matches numpy ?", np.allclose(C_manual, C_np))
print("outer matches numpy ?", np.allclose(C_outer, C_np))
print("rank(A), rank(B), rank(AB) =", np.linalg.matrix_rank(A), np.linalg.matrix_rank(B), np.linalg.matrix_rank(C_np))

---
## Exercise 4: Dense Layer and Attention Shape Tracking

**Task**: Read a neural-network computation as a chain of matrix operations.

**Requirements**:
- Implement a batched dense layer `Y = X W^T + b`.
- Implement a single-head attention pipeline.
- Print the shape of every intermediate object.
- Verify every row of the attention matrix sums to 1.

**Written part**:
- Explain why `Q K^T` is square when `Q` and `K` have the same number of tokens.

In [ ]:
# === Exercise 4: Scaffold ===

def dense_layer(X, W, b):
    pass  # YOUR CODE HERE


def single_head_attention(Q, K, V):
    pass  # YOUR CODE HERE


# Build small Q, K, V matrices and print all intermediate shapes.

In [ ]:
# === Exercise 4: Solution ===

def dense_layer(X, W, b):
    return X @ W.T + b


def single_head_attention(Q, K, V):
    dk = Q.shape[1]
    S = (Q @ K.T) / np.sqrt(dk)
    A = softmax_rows(S)
    O = A @ V
    return S, A, O


X = np.array([[1.0, 0.5, -1.0], [0.0, 2.0, 1.0]])
W = np.array([[1.0, -1.0, 0.5], [0.0, 2.0, 1.0]])
b = np.array([0.1, -0.2])
Y = dense_layer(X, W, b)
print("X shape", X.shape, "W shape", W.shape, "b shape", b.shape, "Y shape", Y.shape)
print("Y =\n", Y)

Q = np.array([[1.0, 0.0], [0.5, 1.0], [1.0, 1.0], [0.0, 1.0]])
K = np.array([[1.0, 0.0], [1.0, 1.0], [0.0, 1.0], [0.5, 0.5]])
V = np.array([[2.0, 0.0], [1.0, 1.0], [0.0, 2.0], [1.5, 0.5]])
S, A, O = single_head_attention(Q, K, V)
print("\nQ shape", Q.shape, "K shape", K.shape, "V shape", V.shape)
print("S shape", S.shape)
print("A shape", A.shape)
print("O shape", O.shape)
print("row sums of A =", A.sum(axis=1))
print("QK^T symmetric when Q = K ?", np.allclose(Q @ Q.T, (Q @ Q.T).T))

---
## Exercise 5: Inverse and the 2x2 Formula

**Task**: Implement the explicit inverse formula for a 2x2 matrix and verify it.

**Requirements**:
- Implement `inverse_2x2(A)`.
- Raise an error or return `None` if the determinant is zero.
- Verify `A A^{-1} = I` and `A^{-1} A = I` numerically.

**Written part**:
- Explain why determinant zero means inverse cannot exist.

In [ ]:
# === Exercise 5: Scaffold ===

def inverse_2x2(A):
    pass  # YOUR CODE HERE


# Test on one invertible example and one singular example.

In [ ]:
# === Exercise 5: Solution ===

def inverse_2x2(A):
    A = np.asarray(A, dtype=float)
    a, b = A[0, 0], A[0, 1]
    c, d = A[1, 0], A[1, 1]
    det = a * d - b * c
    if abs(det) < 1e-12:
        raise ValueError("matrix is singular")
    return (1.0 / det) * np.array([[d, -b], [-c, a]], dtype=float)


A = np.array([[1.0, 2.0], [3.0, 4.0]])
A_inv = inverse_2x2(A)
print("A^{-1} =\n", A_inv)
print("A A^{-1} =\n", A @ A_inv)
print("A^{-1} A =\n", A_inv @ A)

singular = np.array([[1.0, 2.0], [2.0, 4.0]])
try:
    inverse_2x2(singular)
except ValueError as e:
    print("singular example correctly failed:", e)

---
## Exercise 6: Solve vs Explicit Inverse and Conditioning

**Task**: Compare solving a system directly with using an explicit inverse, and inspect the role of condition number.

**Requirements**:
- Solve `A x = b` with both `np.linalg.solve` and `np.linalg.inv(A) @ b`.
- Compute the condition number of a well-conditioned and an ill-conditioned matrix.
- Show that a tiny perturbation to `b` causes a much bigger change in the solution when the matrix is ill-conditioned.

**Written part**:
- Explain why ill-conditioning is a geometry problem, not just a floating-point problem.

In [ ]:
# === Exercise 6: Scaffold ===

# Choose one well-conditioned matrix and one nearly singular matrix.
# Compare solve vs inverse and inspect sensitivity to perturbations.

In [ ]:
# === Exercise 6: Solution ===

A_good = np.array([[3.0, 1.0], [1.0, 2.0]])
b_good = np.array([1.0, 0.0])
print("good matrix condition number =", np.linalg.cond(A_good))
print("solve =", np.linalg.solve(A_good, b_good))
print("inv @ b =", np.linalg.inv(A_good) @ b_good)

A_bad = np.array([[1.0, 1.0], [1.0, 1.0001]])
b_bad = np.array([2.0, 2.0001])
x_bad = np.linalg.solve(A_bad, b_bad)
x_bad_perturbed = np.linalg.solve(A_bad, b_bad + np.array([0.0, 1e-4]))
print("\nbad matrix condition number =", np.linalg.cond(A_bad))
print("x_bad =", x_bad)
print("x_bad_perturbed =", x_bad_perturbed)
print("solution change =", x_bad_perturbed - x_bad)

---
## Exercise 7: Pseudo-Inverse and Least Squares Geometry

**Task**: Use the pseudo-inverse to solve an overdetermined system and verify the projection geometry.

**Requirements**:
- Compute `x* = A^+ b`.
- Compute the projection `A x*`.
- Verify the residual is orthogonal to the column space by checking `A^T (b - A x*)`.

**Written part**:
- Explain why `A A^+` behaves like a projection matrix.

In [ ]:
# === Exercise 7: Scaffold ===

# Use the chapter's overdetermined least-squares setup.
# Compute x*, the projection, the residual, and the orthogonality check.

In [ ]:
# === Exercise 7: Solution ===

A = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
b = np.array([1.0, 2.0, 2.5])
A_pinv = np.linalg.pinv(A)
x_star = A_pinv @ b
proj = A @ x_star
residual = b - proj
P = A @ A_pinv

print("A^+ =\n", A_pinv)
print("x* =", x_star)
print("projection A x* =", proj)
print("residual =", residual)
print("A^T residual =", A.T @ residual)
print("P^2 == P ?", np.allclose(P @ P, P))
print("P^T == P ?", np.allclose(P.T, P))

---
## Exercise 8: Determinant and Row Operations

**Task**: Verify determinant identities experimentally.

**Requirements**:
- Compute determinant of a matrix.
- Swap two rows and show the sign flips.
- Add a multiple of one row to another and show the determinant is unchanged.
- Verify `det(AB) = det(A) det(B)` on one example.

**Written part**:
- Explain why determinant zero means the map collapses volume.

In [ ]:
# === Exercise 8: Scaffold ===

# Pick a 3x3 matrix and test determinant behavior under row operations.

In [ ]:
# === Exercise 8: Solution ===

A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
det_A = np.linalg.det(A)

A_swap = A[[1, 0, 2], :]
A_row_add = A.copy()
A_row_add[1, :] = A_row_add[1, :] + 2 * A_row_add[0, :]

B = np.array([[1.0, 2.0, 0.0], [0.0, 1.0, 1.0], [1.0, 0.0, 1.0]])

print("det(A) =", det_A)
print("det(row-swapped A) =", np.linalg.det(A_swap))
print("sign flipped?", np.allclose(np.linalg.det(A_swap), -det_A))
print("det(row-added A) =", np.linalg.det(A_row_add))
print("unchanged?", np.allclose(np.linalg.det(A_row_add), det_A))
print("det(AB) =", np.linalg.det(A @ B))
print("det(A)det(B) =", np.linalg.det(A) * np.linalg.det(B))

---
## Exercise 9: LU Factorization and Triangular Solves

**Task**: Factor a matrix into `L` and `U`, then solve a system by forward and back substitution.

**Requirements**:
- Use `lu_no_pivot` from the setup or implement your own.
- Solve `L y = b` and `U x = y`.
- Verify `L U = A`.

**Written part**:
- Explain why LU is especially useful when multiple right-hand sides share the same matrix `A`.

In [ ]:
# === Exercise 9: Scaffold ===

# Factor A into L and U, then solve A x = b through two triangular systems.

In [ ]:
# === Exercise 9: Solution ===

A = np.array([[2.0, 1.0, 1.0], [4.0, 3.0, 3.0], [8.0, 7.0, 9.0]])
b = np.array([1.0, 1.0, 1.0])
L, U = lu_no_pivot(A)
y = forward_substitution(L, b)
x = back_substitution(U, y)

print("L =\n", L)
print("U =\n", U)
print("L @ U == A ?", np.allclose(L @ U, A))
print("y =", y)
print("x =", x)
print("A x =", A @ x)
print("Another RHS could reuse the same L and U with only O(n^2) solve cost.")

---
## Exercise 10: QR and Orthonormal Least Squares

**Task**: Solve a least-squares problem using QR.

**Requirements**:
- Compute reduced QR decomposition.
- Verify `Q^T Q = I`.
- Solve `R x = Q^T b`.
- Compare with `np.linalg.lstsq`.

**Written part**:
- Explain why QR is often numerically preferred over normal equations.

In [ ]:
# === Exercise 10: Scaffold ===

# Build a least-squares example and solve it through QR.

In [ ]:
# === Exercise 10: Solution ===

A = np.array([[1.0, 1.0], [1.0, 2.0], [1.0, 3.0]])
b = np.array([1.0, 2.0, 2.0])
Q, R = np.linalg.qr(A, mode='reduced')
x_qr = np.linalg.solve(R, Q.T @ b)
x_lstsq, *_ = np.linalg.lstsq(A, b, rcond=None)

print("Q^T Q =\n", Q.T @ Q)
print("R =\n", R)
print("x via QR =", x_qr)
print("x via lstsq =", x_lstsq)
print("solutions agree?", np.allclose(x_qr, x_lstsq))

---
## Exercise 11: Eigendecomposition and Matrix Powers

**Task**: Compute an eigendecomposition and use it to evaluate a matrix power.

**Requirements**:
- Use a symmetric matrix so the eigen-structure is clean.
- Reconstruct the matrix from its eigendecomposition.
- Compute `A^4` using eigenvalues instead of repeated multiplication.
- Compute the condition number from eigenvalues for the SPD case.

**Written part**:
- Explain why symmetric matrices are especially pleasant for spectral analysis.

In [ ]:
# === Exercise 11: Scaffold ===

# Use A = [[2,1],[1,2]] or another symmetric example.
# Compute eigenvalues, eigenvectors, reconstruction, and A^4.

In [ ]:
# === Exercise 11: Solution ===

A = np.array([[2.0, 1.0], [1.0, 2.0]])
eigvals, eigvecs = np.linalg.eigh(A)
A_reconstructed = eigvecs @ np.diag(eigvals) @ eigvecs.T
A4 = eigvecs @ np.diag(eigvals**4) @ eigvecs.T
kappa = eigvals.max() / eigvals.min()

print("eigenvalues =", eigvals)
print("eigenvectors =\n", eigvecs)
print("reconstruction =\n", A_reconstructed)
print("A^4 via eigendecomposition =\n", A4)
print("condition number =", kappa)
print("positive definite?", np.all(eigvals > 0))

---
## Exercise 12: SVD, Low Rank, LoRA, and Cholesky

**Task**: End with the most AI-relevant matrix ideas: low-rank structure and SPD factorization.

**Requirements**:
- Compute the singular values of a matrix and form the best rank-1 approximation.
- Build a LoRA-style update `DeltaW = B A` and inspect its rank.
- Perform a Cholesky factorization of an SPD matrix and verify `L L^T = A`.

**Written part**:
- Explain why rapidly decaying singular values make low-rank adaptation plausible.

In [ ]:
# === Exercise 12: Scaffold ===

# 1. Compute the SVD of a small matrix and keep only rank 1.
# 2. Build a LoRA-style low-rank update and check its rank.
# 3. Factor an SPD matrix with Cholesky.

In [ ]:
# === Exercise 12: Solution ===

M = np.array([[3.0, 2.0, 2.0], [2.0, 3.0, -2.0]])
U, S, Vt = np.linalg.svd(M, full_matrices=False)
M1 = U[:, :1] @ np.diag(S[:1]) @ Vt[:1, :]
print("singular values =", S)
print("best rank-1 approximation =\n", M1)
print("Frobenius error =", np.linalg.norm(M - M1, ord='fro'))

d = 8
r = 2
B = np.random.randn(d, r)
A_factor = np.random.randn(r, d)
DeltaW = B @ A_factor
print("\nLoRA-style update shape =", DeltaW.shape)
print("rank(DeltaW) <= r ?", np.linalg.matrix_rank(DeltaW) <= r)
print("singular values of DeltaW =", np.linalg.svd(DeltaW, compute_uv=False))

SPD = np.array([[4.0, 2.0, 2.0], [2.0, 5.0, 1.0], [2.0, 1.0, 3.0]])
L = np.linalg.cholesky(SPD)
print("\nL =\n", L)
print("L L^T =\n", L @ L.T)
print("matches SPD ?", np.allclose(L @ L.T, SPD))

---
## Solution Checks

This final cell does a light sanity pass over a few core identities used throughout the notebook.

In [ ]:
print("Running solution checks...\n")

A = np.array([[1., 2.], [3., 4.]])
B = np.array([[5., 6.], [7., 8.]])
assert np.allclose((A @ B).T, B.T @ A.T)

A2 = np.array([[1., 2.], [3., 4.]])
A2_inv = inverse_2x2(A2)
assert np.allclose(A2 @ A2_inv, np.eye(2))

A3 = np.array([[1., 2., 3.], [4., 5., 6.]])
B3 = np.array([[1., 0.], [0., 1.], [1., 1.]])
assert np.allclose(matmul_manual(A3, B3), A3 @ B3)
assert np.allclose(matmul_outer(A3, B3), A3 @ B3)

A_ls = np.array([[1., 2.], [3., 4.], [5., 6.]])
P = A_ls @ np.linalg.pinv(A_ls)
assert np.allclose(P @ P, P)

SPD = np.array([[4., 2., 2.], [2., 5., 1.], [2., 1., 3.]])
L = np.linalg.cholesky(SPD)
assert np.allclose(L @ L.T, SPD)

print("All core checks passed.")